# Chapitre 4 — Partie 4 : Validation Croisée

**Durée estimée : 2h**

## 🎯 Objectifs d'apprentissage

À la fin de cette partie, vous serez capable de :
1. **Expliquer** pourquoi un seul split train/test peut être insuffisant
2. **Implémenter** le K-Fold Cross-Validation avec scikit-learn
3. **Utiliser** Stratified K-Fold pour les problèmes de classification déséquilibrés
4. **Interpréter** les résultats de la validation croisée

---

## Lien avec les chapitres précédents

Dans le **Chapitre 2 (Leçon 2)**, nous avons appris à séparer nos données en train/test avec `train_test_split`. Nous avons aussi évoqué l'idée d'un **validation set** pour tuner les hyperparamètres.

Maintenant, nous allons découvrir une technique plus robuste : la **validation croisée**, qui permet d'évaluer un modèle de manière plus fiable.

---

## 🌍 Problème Réel : Le coup de chance du split

Vous développez un modèle de prédiction. Avec un split 80/20, vous obtenez une accuracy de **92%** sur le test set. Excellent !

Mais votre collègue relance le même code avec `random_state=123` au lieu de `42`... et obtient **84%**.

Un autre essaie avec `random_state=999`... et obtient **89%**.

**Question :** Quelle est la "vraie" performance de votre modèle ?

*(Réponse attendue : Impossible à dire ! Chaque split donne un résultat différent. La performance dépend de quels exemples tombent dans train vs test.)*

---

C'est le problème de la **variabilité du split**. La validation croisée résout ce problème en faisant **plusieurs splits**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

# Créer un dataset
np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10,
                           n_redundant=5, random_state=42)

print("📊 Dataset créé")
print(f"  Taille : {X.shape[0]} exemples, {X.shape[1]} features")
print(f"  Classes : {np.bincount(y)}")

### Démonstration : la variabilité du split unique

In [ ]:
# Tester plusieurs random_state
model = LogisticRegression(max_iter=1000, random_state=42)

scores = []
for rs in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=rs)
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    scores.append(score)
    
print("📊 Variabilité selon le random_state")
print("=" * 50)
for i, s in enumerate(scores):
    print(f"  random_state={i} : {s:.1%}")

print(f"\n  Minimum : {min(scores):.1%}")
print(f"  Maximum : {max(scores):.1%}")
print(f"  Écart   : {max(scores) - min(scores):.1%} ← Variabilité importante !")

---

## 4.1 K-Fold Cross-Validation : L'Idée

Au lieu d'un seul split, on fait **K splits différents** et on moyenne les résultats.

```
┌─────────────────────────────────────────────────────────────────────┐
│              K-FOLD CROSS-VALIDATION (K=5)                          │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   Les données sont divisées en 5 "folds" (plis)                    │
│                                                                     │
│   Fold 1 : [TEST]  [TRAIN] [TRAIN] [TRAIN] [TRAIN] → Score 1       │
│   Fold 2 : [TRAIN] [TEST]  [TRAIN] [TRAIN] [TRAIN] → Score 2       │
│   Fold 3 : [TRAIN] [TRAIN] [TEST]  [TRAIN] [TRAIN] → Score 3       │
│   Fold 4 : [TRAIN] [TRAIN] [TRAIN] [TEST]  [TRAIN] → Score 4       │
│   Fold 5 : [TRAIN] [TRAIN] [TRAIN] [TRAIN] [TEST]  → Score 5       │
│                                                                     │
│   Score Final = Moyenne(Score 1, Score 2, Score 3, Score 4, Score 5)│
│                                                                     │
│   AVANTAGES :                                                       │
│   • Chaque exemple est utilisé pour le test exactement 1 fois      │
│   • Chaque exemple est utilisé pour l'entraînement K-1 fois        │
│   • Le score moyen est plus stable et représentatif                 │
│   • L'écart-type indique la variabilité                             │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

**Question :** Avec K=5, quel pourcentage des données est utilisé pour l'entraînement à chaque fold ?

*(Réponse attendue : 80% (4 folds sur 5) — similaire à un split 80/20, mais répété 5 fois de manière différente.)*

---

## 4.2 Implémentation avec cross_val_score

In [ ]:
# Cross-validation avec cross_val_score
model = LogisticRegression(max_iter=1000, random_state=42)

# K=5 folds
scores_cv = cross_val_score(model, X, y, cv=5)

print("📊 5-Fold Cross-Validation")
print("=" * 50)
for i, score in enumerate(scores_cv, 1):
    print(f"  Fold {i} : {score:.4f}")

print(f"\n  Moyenne : {scores_cv.mean():.4f}")
print(f"  Écart-type : {scores_cv.std():.4f}")
print(f"\n💡 Format standard : {scores_cv.mean():.2%} ± {scores_cv.std():.2%}")

### Interprétation

```
┌─────────────────────────────────────────────────────────────────────┐
│           COMMENT INTERPRÉTER LES RÉSULTATS                         │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   Score = 85.2% ± 2.1%                                              │
│                                                                     │
│   • 85.2% = Performance attendue du modèle                         │
│   • ± 2.1% = Variabilité (1 écart-type)                            │
│                                                                     │
│   Intervalle de confiance approximatif (95%) :                      │
│   85.2% ± 2×2.1% = [81.0%, 89.4%]                                   │
│                                                                     │
│   Si l'écart-type est GRAND :                                       │
│   • Le modèle est instable                                          │
│   • Ou les données sont très hétérogènes                           │
│                                                                     │
│   Si l'écart-type est PETIT :                                       │
│   • Le modèle est stable et fiable                                  │
│   • La moyenne est représentative                                   │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

<details>
<summary>🤔 Question Socratique : Pourquoi utilise-t-on souvent K=5 ou K=10 ?</summary>

### 🔑 Réponse

C'est un compromis entre **fiabilité** et **coût computationnel** :

| K | Train size | Avantages | Inconvénients |
|---|------------|-----------|---------------|
| 2 | 50% | Rapide | Estimateur très variable |
| 5 | 80% | Bon compromis | Standard industriel |
| 10 | 90% | Plus précis | 2× plus lent que K=5 |
| N (LOO) | N-1 | Biais minimal | Très coûteux, haute variance |

**Recommandations :**
- K=5 : datasets moyens, exploration rapide
- K=10 : datasets plus grands, évaluation finale
- LOO (Leave-One-Out) : petits datasets (<100 exemples)

</details>

---

## 4.3 Visualiser le processus K-Fold

In [ ]:
# Visualiser comment KFold divise les données
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

fig, axes = plt.subplots(5, 1, figsize=(12, 6))

for fold_idx, (train_idx, test_idx) in enumerate(kfold.split(X)):
    ax = axes[fold_idx]
    
    # Créer un array de couleurs
    colors = np.zeros(len(X))
    colors[test_idx] = 1  # Test = 1 (orange)
    
    ax.scatter(range(len(X)), [0]*len(X), c=colors, cmap='coolwarm', s=2)
    ax.set_ylabel(f'Fold {fold_idx+1}')
    ax.set_yticks([])
    ax.set_xlim(0, len(X))
    
    # Stats
    ax.text(len(X)+10, 0, f'Train: {len(train_idx)}, Test: {len(test_idx)}', va='center')

axes[0].set_title('5-Fold Cross-Validation : Distribution Train (bleu) / Test (orange)')
axes[-1].set_xlabel('Index des exemples')
plt.tight_layout()
plt.show()

---

## 4.4 Stratified K-Fold : Préserver les Proportions

### Le problème avec les classes déséquilibrées

Avec un K-Fold classique sur des données déséquilibrées (ex: 90% négatifs, 10% positifs), certains folds pourraient avoir très peu (ou pas !) d'exemples de la classe minoritaire.

**Solution :** Le **Stratified K-Fold** garantit que chaque fold a la même proportion de classes que le dataset complet.

In [ ]:
# Créer un dataset déséquilibré
np.random.seed(42)
X_imb, y_imb = make_classification(n_samples=1000, n_features=20, n_informative=10,
                                    weights=[0.9, 0.1], random_state=42)

print("📊 Dataset déséquilibré")
print(f"  Classe 0 : {sum(y_imb == 0)} ({sum(y_imb == 0)/len(y_imb):.1%})")
print(f"  Classe 1 : {sum(y_imb == 1)} ({sum(y_imb == 1)/len(y_imb):.1%})")

In [ ]:
# Comparer KFold vs StratifiedKFold
print("\n📊 Comparaison : KFold vs StratifiedKFold")
print("=" * 60)

# KFold classique
print("\n▶ KFold (sans stratification) :")
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
for i, (train_idx, test_idx) in enumerate(kfold.split(X_imb), 1):
    pct_class1 = y_imb[test_idx].mean() * 100
    print(f"  Fold {i} : {pct_class1:.1f}% de classe 1 dans le test")

# StratifiedKFold
print("\n▶ StratifiedKFold (avec stratification) :")
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for i, (train_idx, test_idx) in enumerate(skfold.split(X_imb, y_imb), 1):
    pct_class1 = y_imb[test_idx].mean() * 100
    print(f"  Fold {i} : {pct_class1:.1f}% de classe 1 dans le test")

print("\n💡 StratifiedKFold préserve les proportions dans chaque fold !")

### Utiliser StratifiedKFold avec cross_val_score

In [ ]:
# cross_val_score utilise automatiquement StratifiedKFold pour la classification !
model = LogisticRegression(max_iter=1000, random_state=42)

# Par défaut, cross_val_score utilise StratifiedKFold pour les classifieurs
scores_stratified = cross_val_score(model, X_imb, y_imb, cv=5)

print("📊 Stratified 5-Fold CV sur données déséquilibrées")
print("=" * 55)
print(f"Score : {scores_stratified.mean():.2%} ± {scores_stratified.std():.2%}")

---

┌─────────────────────────────────────────────────────────────────────┐
│ 📖 DÉFINITION : Stratified K-Fold Cross-Validation                  │
│                                                                     │
│ La validation croisée stratifiée garantit que chaque fold          │
│ contient approximativement la même proportion de chaque classe     │
│ que le dataset complet.                                             │
│                                                                     │
│ Essentiel pour :                                                    │
│ • Classes déséquilibrées (90/10, 99/1, etc.)                       │
│ • Classification multiclasse                                        │
│ • Petits datasets                                                   │
│                                                                     │
│ Note : cross_val_score l'utilise automatiquement pour les          │
│ classifieurs. Pour forcer un comportement spécifique, passez       │
│ explicitement cv=StratifiedKFold(n_splits=5).                      │
└─────────────────────────────────────────────────────────────────────┘

---

## 4.5 Choisir la Métrique avec cross_val_score

In [ ]:
# Par défaut, cross_val_score utilise l'accuracy pour la classification
# Mais on peut spécifier d'autres métriques avec le paramètre 'scoring'

model = LogisticRegression(max_iter=1000, random_state=42)

# Différentes métriques
metriques = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

print("📊 Cross-validation avec différentes métriques")
print("=" * 55)

for nom, scoring in metriques.items():
    scores = cross_val_score(model, X_imb, y_imb, cv=5, scoring=scoring)
    print(f"  {nom:12s} : {scores.mean():.2%} ± {scores.std():.2%}")

---

## 4.6 Comparer Plusieurs Modèles avec CV

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Définir les modèles à comparer
modeles = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree (depth=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
}

print("📊 Comparaison de modèles avec 5-Fold CV")
print("=" * 60)

resultats = []
for nom, model in modeles.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    resultats.append({
        'Modèle': nom,
        'Moyenne': scores.mean(),
        'Std': scores.std(),
        'Scores': scores
    })
    print(f"  {nom:25s} : {scores.mean():.2%} ± {scores.std():.2%}")

# Trouver le meilleur
meilleur = max(resultats, key=lambda x: x['Moyenne'])
print(f"\n🏆 Meilleur modèle : {meilleur['Modèle']}")

In [ ]:
# Visualiser la comparaison
fig, ax = plt.subplots(figsize=(10, 6))

noms = [r['Modèle'] for r in resultats]
moyennes = [r['Moyenne'] for r in resultats]
stds = [r['Std'] for r in resultats]

bars = ax.barh(noms, moyennes, xerr=stds, capsize=5, color='steelblue', edgecolor='black')
ax.set_xlabel('Accuracy')
ax.set_title('Comparaison des modèles (5-Fold CV)')
ax.set_xlim(0.7, 1.0)

# Ajouter les valeurs
for bar, mean, std in zip(bars, moyennes, stds):
    ax.text(mean + std + 0.01, bar.get_y() + bar.get_height()/2,
            f'{mean:.1%} ± {std:.1%}', va='center')

plt.tight_layout()
plt.show()

---

## 4.7 Bonnes Pratiques

```
┌─────────────────────────────────────────────────────────────────────┐
│           BONNES PRATIQUES - VALIDATION CROISÉE                     │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   ✅ À FAIRE                                                        │
│   ────────────                                                      │
│   • Utiliser K=5 ou K=10 (standard industriel)                     │
│   • Toujours reporter moyenne ET écart-type                         │
│   • Utiliser StratifiedKFold pour la classification                │
│   • Utiliser shuffle=True avec random_state fixe                   │
│   • Comparer les modèles sur le MÊME split (même random_state)     │
│                                                                     │
│   ❌ À ÉVITER                                                       │
│   ─────────────                                                     │
│   • Reporter uniquement la moyenne sans écart-type                  │
│   • Utiliser K=2 (trop variable)                                   │
│   • Oublier la stratification pour classes déséquilibrées          │
│   • Faire du preprocessing AVANT le split CV (data leakage !)      │
│                                                                     │
│   ⚠️ ATTENTION AU DATA LEAKAGE                                     │
│   ───────────────────────────────                                  │
│   Le preprocessing (scaling, encoding, feature selection) doit     │
│   être fait DANS chaque fold, pas avant la CV.                     │
│   → Utiliser Pipeline avec cross_val_score (voir Partie 5)         │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

---

## 🧪 Exercice Pratique : Évaluer un Modèle de Churn

Vous travaillez sur un modèle de prédiction de churn (départ de clients).

In [ ]:
# Données de l'exercice
np.random.seed(42)

# Dataset de churn (déséquilibré : 15% de churn)
X_churn, y_churn = make_classification(
    n_samples=2000, n_features=15, n_informative=8,
    weights=[0.85, 0.15], random_state=42
)

print("📊 Dataset Churn")
print("=" * 50)
print(f"  Clients fidèles : {sum(y_churn == 0)} ({sum(y_churn == 0)/len(y_churn):.1%})")
print(f"  Churners        : {sum(y_churn == 1)} ({sum(y_churn == 1)/len(y_churn):.1%})")

### Votre mission :

1. Utilisez une 5-Fold Stratified CV pour évaluer un Random Forest
2. Calculez les scores pour : accuracy, recall, f1, roc_auc
3. Quelle métrique devrait être prioritaire pour ce problème de churn ?
4. Comparez Random Forest à Logistic Regression

In [ ]:
# 🎯 À VOUS DE JOUER !

# Étape 1 & 2 : CV avec différentes métriques
# ...

# Étape 3 : Analyse de la métrique prioritaire
# ...

# Étape 4 : Comparaison
# ...

### 🔑 Solution

In [ ]:
# Solution - Étape 1 & 2 : CV avec différentes métriques
rf = RandomForestClassifier(n_estimators=100, random_state=42)

metriques = ['accuracy', 'recall', 'f1', 'roc_auc']

print("📊 Random Forest - 5-Fold Stratified CV")
print("=" * 55)

for m in metriques:
    scores = cross_val_score(rf, X_churn, y_churn, cv=5, scoring=m)
    print(f"  {m:12s} : {scores.mean():.2%} ± {scores.std():.2%}")

In [ ]:
# Solution - Étape 3 : Analyse
print("\n📋 Analyse : Quelle métrique prioritaire ?")
print("=" * 55)
print("""
Pour la prédiction de CHURN, le RECALL est crucial !

Pourquoi ?
• Un client qui churn et qu'on ne détecte pas = perte de revenus
• Un client fidèle qu'on contacte par erreur = juste un appel inutile

→ Mieux vaut contacter quelques clients fidèles inutilement
  que de laisser partir des churners sans réagir.

Ordre de priorité : RECALL > F1 > ROC-AUC > Accuracy
""")

In [ ]:
# Solution - Étape 4 : Comparaison
lr = LogisticRegression(max_iter=1000, random_state=42)

print("\n📊 Comparaison RF vs Logistic Regression (focus sur Recall)")
print("=" * 60)

for nom, model in [('Random Forest', rf), ('Logistic Regression', lr)]:
    recall_scores = cross_val_score(model, X_churn, y_churn, cv=5, scoring='recall')
    f1_scores = cross_val_score(model, X_churn, y_churn, cv=5, scoring='f1')
    print(f"\n{nom}:")
    print(f"  Recall : {recall_scores.mean():.2%} ± {recall_scores.std():.2%}")
    print(f"  F1     : {f1_scores.mean():.2%} ± {f1_scores.std():.2%}")

---

## 🧠 Réflexion Métacognitive

Avant de passer à la suite :

1. **Pourquoi** la validation croisée donne-t-elle une estimation plus fiable qu'un seul split ?

2. **Dans quel cas** utiliseriez-vous K=10 plutôt que K=5 ?

3. **Que signifie** un écart-type élevé dans les résultats de CV ?

---

## 📝 Résumé

| Concept | Description |
|---------|-------------|
| **K-Fold CV** | Divise les données en K parties, entraîne K fois |
| **Stratified K-Fold** | Préserve les proportions de classes dans chaque fold |
| **K recommandé** | 5 (standard) ou 10 (plus précis) |
| **Format** | Moyenne ± Écart-type (ex: 85% ± 2%) |

**Code essentiel :**
```python
from sklearn.model_selection import cross_val_score, StratifiedKFold

# 5-Fold CV (stratified par défaut pour classification)
scores = cross_val_score(model, X, y, cv=5)
print(f"{scores.mean():.2%} ± {scores.std():.2%}")

# Avec métrique spécifique
scores = cross_val_score(model, X, y, cv=5, scoring='f1')

# Contrôle explicite
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv)
```

---

## ➡️ Prochaine partie

Dans la **Partie 5 : Optimisation des Hyperparamètres**, nous allons apprendre à trouver les meilleurs hyperparamètres avec GridSearchCV et RandomizedSearchCV.

**Question de transition :** Maintenant qu'on sait évaluer un modèle de manière robuste, comment trouver automatiquement les meilleurs hyperparamètres ?